In [ ]:
# 데이터늘리기
# 한글토크나이져사용
# tfidfvectors도 사용
# 목표... 다음리뷰의 데이터로 train test 분리해서 최대한 test의 감정분류를 높여보기
# 모델에서 cnn대신 RNN LSTM도 적용해서 각 모델별 성능의 차이 

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from gensim.models import Word2Vec  # 사전 학습된 모델

# ==========================================
# 1. 데이터 준비
# ==========================================

data = {
    'reviews': [
        '이 영화 정말 좋아 최고야',
        '시간 아까운 쓰레기 영화',
        '배우들 연기가 너무 훌륭해요',
        '스토리가 지루하고 뻔하다',
        '인생 최고의 명작입니다 추천',
        '돈 주고 보기 아까운 졸작'
    ],
    'ratings': [5, 1, 4, 2, 5, 1]
}

df = pd.DataFrame(data)

# 평점 이진화
# 4~5점 -> 긍정(1)
# 1~3점 -> 부정(0)

df['ratings'] = df['ratings'].apply(
    lambda x: 1 if x >= 4 else 0
)
# print(df)
# ==========================================
# 2. 토큰화
# ==========================================
def tokenize(text):
    return text.split()

tokenized_sentences = [
    tokenize(review)
    for review in df['reviews']
]

# print(tokenized_sentences)
# ==========================================
# 3. Word2Vec 사전학습 (전이학습용)
# ==========================================
EMBED_DIM = 100
w2v_model = Word2Vec(
    sentences=tokenized_sentences,
    vector_size=EMBED_DIM,
    window=3,
    min_count=1,
    workers=4
)
print("Word2Vec 사전학습 완료")

# ==========================================
# 4. Vocabulary 생성
# ==========================================
vocab = {
    '<PAD>': 0,
    '<UNK>': 1
}
for sentence in tokenized_sentences:
    for word in sentence:
        if word not in vocab:
            vocab[word] = len(vocab)

# print(vocab)
VOCAB_SIZE = len(vocab)
# ==========================================
# 5. 문장 -> 숫자 변환
# ==========================================
MAX_LEN = 10
def text_to_sequence(text, vocab, max_len):
    seq = [
        vocab.get(word, vocab['<UNK>'])
        for word in tokenize(text)
    ]
    # padding
    if len(seq) < max_len:
        seq += [vocab['<PAD>']] * (max_len - len(seq))
    return seq[:max_len]

df['sequence'] = df['reviews'].apply(
    lambda x: text_to_sequence(x, vocab, MAX_LEN)
)

# print(df['sequence'])

# ==========================================
# 6. Embedding Matrix 생성
# ==========================================
embedding_matrix = np.random.normal(
    scale=0.01,
    size=(VOCAB_SIZE, EMBED_DIM)
)

# PAD 벡터는 0으로 유지
embedding_matrix[0] = np.zeros((EMBED_DIM,))

for word, idx in vocab.items():
    if word in ['<PAD>', '<UNK>']:
        continue

    if word in w2v_model.wv:
        embedding_matrix[idx] = w2v_model.wv[word]

pretrained_weight = torch.FloatTensor(
    embedding_matrix
)
# print(pretrained_weight.shape)

# ==========================================
# 7. Dataset
# ==========================================
class ReviewDataset(Dataset):
    def __init__(self, sequences, labels):
        self.x = torch.LongTensor(sequences)
        self.y = torch.FloatTensor(labels)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

dataset = ReviewDataset(
    df['sequence'].tolist(),
    df['ratings'].tolist()
)

dataloader = DataLoader(
    dataset,
    batch_size=2,
    shuffle=True
)

# ==========================================
# 8. TextCNN 모델
# ==========================================

class TextCNN(nn.Module):

    def __init__(
        self,
        vocab_size,
        embed_dim,
        filter_sizes,
        num_filters,
        pretrained_weight=None,
        freeze_emb=False
    ):
        super().__init__()
        # 전이학습 Embedding
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=0
        )

        # pretrained weight 적용
        if pretrained_weight is not None:
            self.embedding.weight.data.copy_(
                pretrained_weight
            )
            # freeze 여부
            self.embedding.weight.requires_grad = (
                not freeze_emb
            )

        # CNN
        self.convs = nn.ModuleList([
            nn.Conv2d(
                in_channels=1,
                out_channels=num_filters,
                kernel_size=(fs, embed_dim)
            )
            for fs in filter_sizes
        ])
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(
            len(filter_sizes) * num_filters,
            1
        )

    def forward(self, x):
        # (batch, seq_len)
        x = self.embedding(x)
        # (batch, seq_len, embed_dim)
        x = x.unsqueeze(1)
        # (batch, 1, seq_len, embed_dim)
        pooled_outputs = []
        for conv in self.convs:
            # convolution
            c = F.relu(conv(x))
            # (batch, num_filters, H, 1)
            c = c.squeeze(3)
            # (batch, num_filters, H)
            # max pooling
            p = F.max_pool1d(
                c,
                kernel_size=c.size(2)
            )
            p = p.squeeze(2)
            pooled_outputs.append(p)

        # concat
        x = torch.cat(
            pooled_outputs,
            dim=1
        )

        x = self.dropout(x)
        logits = self.fc(x)
        return logits.squeeze(1)

# ==========================================
# 9. 모델 생성
# ==========================================

FILTER_SIZES = [2, 3, 4]
NUM_FILTERS = 20

model = TextCNN(
    vocab_size=VOCAB_SIZE,
    embed_dim=EMBED_DIM,
    filter_sizes=FILTER_SIZES,
    num_filters=NUM_FILTERS,
    pretrained_weight=pretrained_weight,
    freeze_emb=False
)

print(model)

# ==========================================
# 10. Loss / Optimizer
# ==========================================

criterion = nn.BCEWithLogitsLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

# ==========================================
# 11. 학습
# ==========================================

EPOCHS = 10

print("\n학습 시작\n")
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for batch_x, batch_y in dataloader:
        optimizer.zero_grad()
        logits = model(batch_x)
        loss = criterion(
            logits,
            batch_y
        )
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(dataloader)
    print(
        f"Epoch: {epoch+1:02d} "
        f"Loss: {avg_loss:.4f}"
    )

print("\n학습 완료")

# ==========================================
# 12. 추론
# ==========================================

def predict(text):
    model.eval()
    sequence = text_to_sequence(
        text,
        vocab,
        MAX_LEN
    )

    x = torch.LongTensor(sequence).unsqueeze(0)
    with torch.no_grad():
        logits = model(x)
        prob = torch.sigmoid(logits)
        pred = (prob >= 0.5).float()

    print(f"\n리뷰: {text}")
    print(f"긍정확률: {prob.item():.4f}")

    if pred.item() == 1:
        print("예측: 긍정 ")
    else:
        print("예측: 부정 ")

Word2Vec 사전학습 완료
TextCNN(
  (embedding): Embedding(25, 100, padding_idx=0)
  (convs): ModuleList(
    (0): Conv2d(1, 20, kernel_size=(2, 100), stride=(1, 1))
    (1): Conv2d(1, 20, kernel_size=(3, 100), stride=(1, 1))
    (2): Conv2d(1, 20, kernel_size=(4, 100), stride=(1, 1))
  )
  (dropout): Dropout(p=0.5, inplace=False)
  (fc): Linear(in_features=60, out_features=1, bias=True)
)

학습 시작

Epoch: 01 Loss: 0.6920
Epoch: 02 Loss: 0.6908
Epoch: 03 Loss: 0.6883
Epoch: 04 Loss: 0.6839
Epoch: 05 Loss: 0.6825
Epoch: 06 Loss: 0.6831
Epoch: 07 Loss: 0.6788
Epoch: 08 Loss: 0.6833
Epoch: 09 Loss: 0.6752
Epoch: 10 Loss: 0.6605

학습 완료


In [5]:
# ==========================================
# 13. 테스트
# ==========================================

predict("배우 연기가 정말 훌륭하다")
predict("돈 아까운 최악의 영화")
predict("이 영화는 재미있고 추천할 만 합니다. 즐겁습니다. 좋습니다.")
predict("재미없는 영화")


리뷰: 배우 연기가 정말 훌륭하다
긍정확률: 0.5010
예측: 긍정 

리뷰: 돈 아까운 최악의 영화
긍정확률: 0.4902
예측: 부정 

리뷰: 이 영화는 재미있고 추천할 만 합니다. 즐겁습니다. 좋습니다.
긍정확률: 0.4973
예측: 부정 

리뷰: 재미없는 영화
긍정확률: 0.4951
예측: 부정 
